# Chapter 15 — The Black–Scholes–Merton Model

Code behind the numbers in the chapter: the formula and its parity check, the
limits it must respect, the normal distribution function it needs, volatility
estimated from history and implied from prices, dividends, and the two ways of
handling an American option.

| Section | What it reproduces |
|---|---|
| 1 | The formula, a call and a put, and parity (Example 15.1, Figure 15.1) |
| 2 | The formula at its edges (Section 15.5) |
| 3 | Computing $N(x)$: the error function against the polynomial, and the far wings (Section 15.5) |
| 4 | Historical volatility and its standard error (Example 15.2, Tables 15.1 and 15.2) |
| 5 | Implied volatility by bisection and by Newton's method (Example 15.3, Figure 15.2) |
| 6 | Two clocks: variance on trading days, interest on calendar days (Section 15.8) |
| 7 | A continuous dividend yield (Example 15.4) |
| 8 | Known cash dividends, and the volatility of $S^*$ (Example 15.5, Question 15.16) |
| 9 | The American put: the European floor against a tree (Section 15.10) |
| 10 | Black's approximation against a tree (Example 15.6) |
| 11 | The skew of Table 15.3, read from the chain of Table 10.2 (Section 15.11) |

Each cell prints the computed value beside the one the chapter prints, to the
decimals the chapter gives, so a figure that has drifted is visible rather than
merely absent. NumPy for the trees, SciPy for a root-finder and an independent
normal distribution, matplotlib for the figures; everything else is the
standard library.

In [ ]:
from math import log, sqrt, exp, erfc, pi

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.stats import norm


def show(label, got, printed, dp=4):
    """Print a computed value beside the one the chapter prints, compared at the chapter's decimals."""
    flag = "ok" if abs(got - printed) <= 0.5 * 10 ** -dp + 1e-12 else "DIFFERS"
    shown = "%14.9f" % got if 0 < abs(printed) < 0.01 else "%14.6f" % got
    print("%-50s %s   book %12s   %s" % (label, shown, "%.*f" % (dp, printed), flag))


def N(x):
    """Standard normal distribution function, from the complementary error function."""
    return 0.5 * erfc(-x / sqrt(2.0))


def bsm(S, K, r, sigma, T, q=0.0):
    """Black-Scholes-Merton d1, d2, call and put, with a continuous yield q."""
    d1 = (log(S / K) + (r - q + 0.5 * sigma * sigma) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    call = S * exp(-q * T) * N(d1) - K * exp(-r * T) * N(d2)
    put = K * exp(-r * T) * N(-d2) - S * exp(-q * T) * N(-d1)
    return d1, d2, call, put


# The book's figure palette, from preamble.tex: exampleframe for the primary series,
# snapshotframe for the contrast one. Series are distinguished by dash pattern as well
# as color, because the book prints in grayscale.
BLUE, AMBER = "#28466E", "#78551E"
plt.rcParams.update({"figure.figsize": (6.4, 3.4), "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})

## 1. The formula, a call and a put — Example 15.1, Figure 15.1

$c = S_0N(d_1) - Ke^{-rT}N(d_2)$, and the put by parity. The chapter works the
example by hand, reading $N$ from a table at $d_1$ rounded to 0.5157, so the
fifth decimal it prints for $N(d_1)$ is the table's; the unrounded $d_1$ gives a
fifth decimal two units lower, and nothing downstream moves.

Then parity, which the formulas satisfy identically because $N(d) + N(-d) = 1$.
A gap is an implementation error, never a modeling question, which is why it is
the first test to run on any pricer. The last line reads $N(d_2)$ as the
risk-neutral probability of exercise (Section 15.4).

In [ ]:
S0, K, r, sigma, T = 58.0, 55.0, 0.04, 0.24, 0.5
d1, d2, c, p = bsm(S0, K, r, sigma, T)
disc_K = K * exp(-r * T)

show("sigma sqrt(T)", sigma * sqrt(T), 0.16971, 5)
show("ln(58/55)", log(S0 / K), 0.053110, 6)
show("(r + sigma^2/2) T", (r + 0.5 * sigma ** 2) * T, 0.034400, 6)
show("d1", d1, 0.5157)
show("d2", d2, 0.3460)
show("N(0.5157), the table lookup", N(0.5157), 0.69697, 5)
show("N(-0.5157)", N(-0.5157), 0.30303, 5)
show("N(0.3460)", N(0.3460), 0.6353)
show("N(-0.3460)", N(-0.3460), 0.3647)
print("N(d1) at the unrounded d1 = %.6f, the delta of Section 15.4" % N(d1))
show("delta N(d1), Section 15.4", N(d1), 0.6970)
print()
show("58 N(d1)", S0 * N(0.5157), 40.42, 2)
show("55 e^{-0.02} N(d2)", disc_K * N(0.3460), 34.25, 2)
show("call", c, 6.17, 2)
show("55 e^{-0.02} N(-d2)", disc_K * N(-0.3460), 19.66, 2)
show("58 N(-d1)", S0 * N(-0.5157), 17.58, 2)
show("put", p, 2.08, 2)
print()
show("call, unrounded", c, 6.1731)
show("put, unrounded", p, 2.0840)
show("c - p", c - p, 4.0891)
show("K e^{-rT}", disc_K, 53.9109)
show("S0 - K e^{-rT}", S0 - disc_K, 4.0891)

gap = max(abs(bsm(s, K, r, v, t)[2] - bsm(s, K, r, v, t)[3] - (s - K * exp(-r * t)))
          for s in (20.0, 58.0, 150.0) for v in (0.02, 0.24, 2.0) for t in (0.01, 0.5, 10.0))
print("largest parity gap over 27 input sets: %.1e, rounding noise only" % gap)
show("N(d2) as an exercise probability, percent", 100 * N(d2), 64, 0)

Figure 15.1 plots the call against the stock price for three maturities, with
$K = 50$, $r = 4\%$, $\sigma = 30\%$. Every one of its 48 plotted points is
recomputed here; the curve lies above intrinsic value by the time value, widest
near the money, and the gap closes as $T$ shrinks.

In [ ]:
FIG_15_1 = {
    0.25: (0.0000, 0.0000, 0.0001, 0.0031, 0.0383, 0.2332, 0.8617, 2.2246, 4.4445, 7.4088,
           10.8833, 14.6470, 18.5512, 22.5155, 26.5032, 30.4993),
    1.0: (0.0046, 0.0356, 0.1560, 0.4729, 1.1097, 2.1674, 3.6990, 5.7059, 8.1495, 10.9679,
          14.0921, 17.4554, 20.9999, 24.6783, 28.4535, 32.2978),
    2.0: (0.1161, 0.3702, 0.8815, 1.7269, 2.9497, 4.5601, 6.5424, 8.8642, 11.4851, 14.3624,
          17.4549, 20.7255, 24.1416, 27.6755, 31.3042, 35.0088),
}
bad = 0
for t, printed in FIG_15_1.items():
    for i, value in enumerate(printed):
        if abs(bsm(20.0 + 4 * i, 50.0, 0.04, 0.30, t)[2] - value) > 0.5e-4:
            bad += 1
show("Figure 15.1 points that reproduce, of 48", 48 - bad, 48, 0)

grid = np.linspace(20.0, 80.0, 241)
fig, ax = plt.subplots()
ax.plot(grid, np.maximum(grid - 50.0, 0.0), color="0.6", ls="--", lw=1.2, label="intrinsic value")
for t, color, ls in ((0.25, BLUE, "-"), (1.0, AMBER, "-."), (2.0, "black", ":")):
    ax.plot(grid, [bsm(s, 50.0, 0.04, 0.30, t)[2] for s in grid], color=color, ls=ls, lw=1.6,
            label="T = %g" % t)
ax.set_xlabel("Stock price S0"); ax.set_ylabel("Call value")
ax.set_title("Figure 15.1: call value against the stock price")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 2. The formula at its edges — Section 15.5

Each limit reproduces a bound Chapter 11 reached with no model at all. The
"limits" below are finite stand-ins (a stock at 5,000, a volatility of 0.01% or
5,000%, a life of a few seconds), which is enough to put the formula on the bound to
the precision printed. The put rows are the call rows plus $Ke^{-rT} - S_0$,
which is parity again.

In [ ]:
def call(S, sig=sigma, t=T):
    return bsm(S, K, r, sig, t)[2]


def put(S, sig=sigma, t=T):
    return bsm(S, K, r, sig, t)[3]


rows = (
    ("call, S0 -> infinity: S0 - K e^{-rT}", call(5000.0), 5000.0 - disc_K),
    ("call, S0 -> 0: zero", call(0.01), 0.0),
    ("call, sigma -> 0, S0 = 58: S0 - K e^{-rT}", call(58.0, sig=1e-4), 58.0 - disc_K),
    ("call, sigma -> 0, S0 = 53 < K e^{-rT}: zero", call(53.0, sig=1e-4), 0.0),
    ("call, sigma -> infinity: S0", call(58.0, sig=50.0), 58.0),
    ("call, T -> 0: max(S0 - K, 0)", call(58.0, t=1e-4 / 365), 3.0),
    ("put, S0 -> infinity: zero", put(5000.0), 0.0),
    ("put, S0 -> 0: K e^{-rT}", put(1e-9), disc_K),
    ("put, sigma -> 0, S0 = 53: K e^{-rT} - S0", put(53.0, sig=1e-4), disc_K - 53.0),
    ("put, sigma -> infinity: K e^{-rT}", put(58.0, sig=50.0), disc_K),
    ("put, T -> 0, S0 = 50: max(K - S0, 0)", put(50.0, t=1e-4 / 365), 5.0),
)
print("%-46s %12s %12s %10s" % ("limit", "formula", "bound", "gap"))
for label, value, bound in rows:
    print("%-46s %12.6f %12.6f %10.1e" % (label, value, bound, value - bound))
print("largest gap: %.1e" % max(abs(v - b) for _, v, b in rows))

## 3. Computing $N(x)$ — Section 15.5

Production code takes $N(x) = \operatorname{erfc}(-x/\sqrt{2})/2$ from the
double-precision complementary error function, which is what `N` above does.
The polynomial the chapter prints is accurate to $7.5 \times 10^{-8}$ in
absolute terms, and the first lines confirm the bound on a fine grid.

The chapter's warning is about *relative* error. Far in a wing, where $N(-d)$ is
around $10^{-6}$, an absolute error of $10^{-8}$ is no longer small, and an
implied volatility read from a deep out-of-the-money price divides that error by
a small vega. The table shows the relative error of the tail growing by three
orders of magnitude from $x = 2$ to $x = 6$, and the implied volatility of
out-of-the-money puts on the Example 15.3 stock drifting further from the true
29.85% as the strike moves away from the money. The drift is still small here;
the point is the direction, and that it is invisible near the money.

In [ ]:
GAMMA = 0.2316419
A = (0.319381530, -0.356563782, 1.781477937, -1.821255978, 1.330274429)


def poly_tail(x):
    """1 - N(x) for x >= 0 from the chapter's polynomial, computed directly rather than by subtraction."""
    k = 1.0 / (1.0 + GAMMA * x)
    return exp(-0.5 * x * x) / sqrt(2.0 * pi) * sum(a * k ** (j + 1) for j, a in enumerate(A))


def N_poly(x):
    return 1.0 - poly_tail(x) if x >= 0 else poly_tail(-x)


xs = np.linspace(0.0, 8.0, 8001)
worst = max(abs(N_poly(x) - N(x)) for x in xs)
print("largest absolute error of the polynomial on [0, 8]: %.2e (chapter: 7.5e-8)" % worst)
print("erfc route against scipy.stats.norm.cdf on the same grid: %.1e"
      % max(abs(N(x) - norm.cdf(x)) for x in xs))
print()
print("%6s %14s %14s %14s" % ("x", "N(-x)", "polynomial", "rel. error"))
for x in (2.0, 3.0, 4.0, 4.75, 5.0, 6.0):
    exact = N(-x)
    print("%6.2f %14.6e %14.6e %14.2e" % (x, exact, poly_tail(x), poly_tail(x) / exact - 1.0))


def put_with(Nf, S, K_, r_, sig, t):
    d1_ = (log(S / K_) + (r_ + 0.5 * sig * sig) * t) / (sig * sqrt(t))
    return K_ * exp(-r_ * t) * Nf(-d1_ + sig * sqrt(t)) - S * Nf(-d1_)


print()
print("%8s %14s %16s %18s" % ("strike", "put price", "implied vol", "error, vol bp"))
for strike in (20.0, 18.0, 16.0, 14.0, 13.0, 12.0):
    price = put_with(N, 21.0, strike, 0.03, 0.2985, 0.25)
    iv = brentq(lambda v: put_with(N_poly, 21.0, strike, 0.03, v, 0.25) - price, 0.05, 2.0, xtol=1e-14)
    print("%8.2f %14.8f %15.6f%% %18.4f" % (strike, price, 100 * iv, 1e4 * (iv - 0.2985)))

## 4. Historical volatility — Example 15.2, Tables 15.1 and 15.2

Ten daily closes, ten log returns, the sample standard deviation, and
annualization by $\sqrt{252}$. Then the example's three cautions as numbers: the
standard error $\sigma/\sqrt{2n}$ and the two-standard-error band it implies;
the overstatement from annualizing with 365 days instead of 252 (a 242-day year,
the count on China's exchanges, is printed alongside for comparison); and the
effect of dropping the mean.

Table 15.2 then prices the standard error: the Example 15.1 call at volatilities
five and ten points either side of 24%, the outer rows sitting roughly two
standard errors out.

In [ ]:
closes = [50.00, 50.75, 50.20, 51.30, 51.05, 50.40, 51.10, 51.85, 51.60, 52.30, 52.00]
u = np.diff(np.log(closes))
printed_u = (0.014889, -0.010897, 0.021676, -0.004885, -0.012814,
             0.013793, 0.014570, -0.004833, 0.013475, -0.005753)
show("Table 15.1 returns that reproduce, of 10",
     sum(abs(a - b) <= 0.5e-6 for a, b in zip(u, printed_u)), 10, 0)
n = len(u)
show("sum u_i", u.sum(), 0.039221, 6)
show("sum u_i^2", (u ** 2).sum(), 0.00163889, 8)
s2 = ((u ** 2).sum() - u.sum() ** 2 / n) / (n - 1)
show("s^2", s2, 0.000165007, 9)
s = sqrt(s2)
show("s per day", s, 0.012846, 6)
show("np.std(u, ddof=1), the same thing", np.std(u, ddof=1), 0.012846, 6)
show("sqrt(252)", sqrt(252), 15.8745)
vol = 100 * s * sqrt(252)
show("annualized volatility, percent", vol, 20.4, 1)

se = 20.4 / sqrt(2 * n)
show("standard error sigma/sqrt(2n), points", se, 4.6, 1)
show("lower end of the 2-standard-error band, percent", 20.4 - 2 * se, 11, 0)
show("upper end of the 2-standard-error band, percent", 20.4 + 2 * se, 30, 0)

for days in (252, 365, 242):
    print("annualized with sqrt(%d) = %.4f: %.2f%%" % (days, sqrt(days), 100 * s * sqrt(days)))
show("overstatement from 365 days, percent", 100 * (sqrt(365 / 252) - 1), 20, 0)

s_zero_mean = sqrt((u ** 2).sum() / n)
show("s with the mean dropped, divided by n", s_zero_mean, 0.012802, 6)
show("change in the annualized figure, points", vol - 100 * s_zero_mean * sqrt(252), 0.1, 1)

In [ ]:
base = bsm(58.0, 55.0, 0.04, 0.24, 0.5)[2]
print("%8s %12s %12s %10s" % ("sigma", "call value", "difference", "percent"))
TABLE_15_2 = ((0.14, 4.8292, -1.3440, -21.8), (0.19, 5.4736, -0.6995, -11.3),
              (0.29, 6.9003, 0.7272, 11.8), (0.34, 7.6428, 1.4697, 23.8))
for sig, value, diff, pct in TABLE_15_2:
    v = bsm(58.0, 55.0, 0.04, sig, 0.5)[2]
    print("%7.0f%% %12.4f %+12.4f %+9.1f%%" % (100 * sig, v, v - base, 100 * (v - base) / base))
    show("  sigma = %.0f%%: value" % (100 * sig), v, value)
    show("  sigma = %.0f%%: difference" % (100 * sig), v - base, diff)
    show("  sigma = %.0f%%: percent" % (100 * sig), 100 * (v - base) / base, pct, 1)
se24 = 24 / sqrt(20)
print("two standard errors at 24%%: +/-%.1f points; the outer rows are %.1f standard errors out"
      % (2 * se24, 10 / se24))

## 5. Implied volatility — Example 15.3, Figure 15.2

The call is strictly increasing in $\sigma$, so the inversion has one answer.
Bisection is slow and certain: it halves a bracket that is known to contain the
root. Newton's method uses the slope, vega, and converges in a handful of steps
from a sensible start. Both land on 29.85%, as does SciPy's `brentq`.

The figure shows why the chapter reads implied volatility from the
out-of-the-money option. Below about 5% the call's value sits on its lower bound
$S_0 - Ke^{-rT}$, so the curve is nearly flat there, and the table that follows
measures what that does: a price error of a tenth of a cent (0.001) moves the
implied volatility by a few hundredths of a point at 30%, by more than a point at
4%, and at 2% the price no longer determines the volatility at all. Switching to
the put at the same strike does not steepen the curve, since its vega is the
same; what changes is that the put's price is all time value, so the whole of it
carries information about volatility. By parity it implies the same 29.85%.

In [ ]:
S_iv, K_iv, r_iv, T_iv, market = 21.0, 20.0, 0.03, 0.25, 1.875

for sig, printed in ((0.20, 1.5148), (0.30, 1.8807), (0.29, 1.8429)):
    show("call at sigma = %.0f%%" % (100 * sig), bsm(S_iv, K_iv, r_iv, sig, T_iv)[2], printed)


def implied_bisection(price, S, K_, r_, t, q=0.0, leg=2, lo=1e-4, hi=5.0, tol=1e-10):
    """Halve [lo, hi] until it is narrower than tol; leg 2 is the call, 3 the put."""
    steps = 0
    while hi - lo > tol:
        mid = 0.5 * (lo + hi)
        if bsm(S, K_, r_, mid, t, q)[leg] < price:
            lo = mid
        else:
            hi = mid
        steps += 1
    return 0.5 * (lo + hi), steps


def vega(S, K_, r_, sig, t, q=0.0):
    d1_ = bsm(S, K_, r_, sig, t, q)[0]
    return S * exp(-q * t) * sqrt(t) * exp(-0.5 * d1_ * d1_) / sqrt(2.0 * pi)


def implied_newton(price, S, K_, r_, t, q=0.0, start=0.30, steps=6):
    path = [start]
    for _ in range(steps):
        sig = path[-1]
        path.append(sig - (bsm(S, K_, r_, sig, t, q)[2] - price) / vega(S, K_, r_, sig, t, q))
    return path


iv_b, n_b = implied_bisection(market, S_iv, K_iv, r_iv, T_iv)
print("bisection: %.8f after %d halvings" % (iv_b, n_b))
print("Newton from 30%: " + ", ".join("%.8f" % v for v in implied_newton(market, S_iv, K_iv, r_iv, T_iv)))
iv_brent = brentq(lambda v: bsm(S_iv, K_iv, r_iv, v, T_iv)[2] - market, 1e-4, 5.0, xtol=1e-12)
show("implied volatility, percent", 100 * iv_brent, 29.85, 2)

lower = S_iv - K_iv * exp(-r_iv * T_iv)
show("lower bound S0 - K e^{-rT}", lower, 1.149, 3)
put_market = market - lower                               # put-call parity
iv_put, _ = implied_bisection(put_market, S_iv, K_iv, r_iv, T_iv, leg=3)
print("the put at the same strike, by parity: price %.4f, implied volatility %.4f%%"
      % (put_market, 100 * iv_put))

print()
print("%8s %12s %16s %22s" % ("sigma", "call", "vega per point", "move for a 0.001 error"))
for sig in (0.02, 0.04, 0.06, 0.10, 0.30):
    vp = vega(S_iv, K_iv, r_iv, sig, T_iv) / 100
    print("%7.0f%% %12.6f %16.6f %19.3f pts" % (100 * sig, bsm(S_iv, K_iv, r_iv, sig, T_iv)[2], vp, 0.001 / vp))

In [ ]:
FIG_15_2 = ((0.5, 1.1494), (2, 1.1494), (4, 1.1497), (6, 1.1566), (8, 1.1789), (10, 1.2160),
            (14, 1.3199), (18, 1.4466), (22, 1.5853), (26, 1.7308), (30, 1.8807), (34, 2.0333),
            (38, 2.1877), (42, 2.3435), (46, 2.5002), (50, 2.6576), (54, 2.8153), (58, 2.9734))
good = sum(abs(bsm(S_iv, K_iv, r_iv, x / 100, T_iv)[2] - v) <= 0.5e-4 for x, v in FIG_15_2)
show("Figure 15.2 points that reproduce, of 18", good, 18, 0)

vols = np.linspace(0.005, 0.60, 300)
fig, ax = plt.subplots()
ax.axhline(lower, color="0.6", ls="--", lw=1.2, label="lower bound S0 - K exp(-rT)")
ax.plot(100 * vols, [bsm(S_iv, K_iv, r_iv, v, T_iv)[2] for v in vols], color=BLUE, lw=1.6,
        label="model value")
ax.axhline(market, color=AMBER, ls="-.", lw=1.4, label="market price 1.875")
ax.axvline(100 * iv_brent, color="0.5", ls=":", lw=1.0)
ax.annotate("%.2f%%" % (100 * iv_brent), (100 * iv_brent, 1.05), ha="center", fontsize=8)
ax.set_xlim(0, 60); ax.set_ylim(1.0, 3.2)
ax.set_xlabel("Volatility (%)"); ax.set_ylabel("Call value")
ax.set_title("Figure 15.2: implied volatility as an inversion")
ax.legend(frameon=False, loc="upper left")
plt.tight_layout(); plt.show()

## 6. Two clocks — Section 15.8

Variance accrues on trading days and interest on calendar days, so $d_1$ carries
$rT_{\text{cal}}$ beside $\sigma^2T_{\text{trd}}/2$. For a 30-calendar-day
option the two fractions are $21/252$ and $30/365$; their ratio, $252/365$ against
$21/30$, is close to one on average, which is why mixing them is an easy error to
miss. The function below keeps them apart, and the last section of this
notebook uses it on the chain of Table 10.2.

In [ ]:
def call_two_clocks(S, K_, r_, sig, t_trading, t_calendar):
    """European call with variance over trading time and discounting over calendar time."""
    v = sig * sqrt(t_trading)
    d1_ = (log(S / K_) + r_ * t_calendar + 0.5 * v * v) / v
    return S * N(d1_) - K_ * exp(-r_ * t_calendar) * N(d1_ - v)


print("252/365 = %.4f against 21/30 = %.4f" % (252 / 365, 21 / 30))
print("30-day option at sigma = 24%%: sd over its life %.5f (21/252) against %.5f (30/365)"
      % (0.24 * sqrt(21 / 252), 0.24 * sqrt(30 / 365)))
for trading_days in (8, 10):
    print("14 calendar days holding %d trading days: variance %+.1f%% against 14/365"
          % (trading_days, 100 * ((trading_days / 252) / (14 / 365) - 1)))

## 7. A continuous dividend yield — Example 15.4

Replace $S_0$ by $S_0e^{-qT}$ throughout. The index at 1,500, struck at 1,550,
with a 2.5% yield; the chapter prints $d_1$, $d_2$ and the normal values to six
decimals, so they are compared at six.

In [ ]:
d1i, d2i, ci, _ = bsm(1500.0, 1550.0, 0.04, 0.18, 0.5, q=0.025)
show("d1", d1i, -0.135056, 6)
show("d2", d2i, -0.262335, 6)
show("N(d1)", N(d1i), 0.446284, 6)
show("N(d2)", N(d2i), 0.396532, 6)
show("1500 e^{-0.0125} N(d1)", 1500 * exp(-0.0125) * N(d1i), 661.11, 2)
show("1550 e^{-0.02} N(d2)", 1550 * exp(-0.02) * N(d2i), 602.45, 2)
show("call", ci, 58.66, 2)
show("one contract at the $100 multiplier", 100 * ci, 5866, 0)
show("risk-neutral chance of finishing above 1,550, percent", 100 * N(d2i), 40, 0)
show("gain needed to reach the strike, percent", 100 * (1550 / 1500 - 1), 3.3, 1)

## 8. Known cash dividends, and the volatility of $S^*$ — Example 15.5, Question 15.16

Subtract the present value of the dividends and price on what is left. The
chapter then notes that the volatility in this calculation belongs to $S^*$, not
to $S$. Appendix A makes that precise: $S^*$ carries the same absolute risk on a
smaller base, so $\sigma^* \approx \sigma\,S/S^* = \sigma/(1 - \text{PV}(D)/S)$.
Its own illustration (a 6% yield, two years) is checked first; then the case
Question 15.16 asks about, a 4% yield over five years, where $\text{PV}(D)/S =
1 - e^{-0.2}$ and the adjustment is about 22%. The last lines show what using
the unadjusted figure does to a five-year at-the-money call: it understates.

In [ ]:
pv1, pv2 = 0.80 * exp(-0.04 * 2 / 12), 0.80 * exp(-0.04 * 5 / 12)
show("PV of the two-month dividend", pv1, 0.7947)
show("PV of the five-month dividend", pv2, 0.7868)
show("PV of dividends", pv1 + pv2, 1.5815)
S_star = 50.0 - pv1 - pv2
show("S*", S_star, 48.4185)
c_div = bsm(S_star, 50.0, 0.04, 0.30, 0.5)[2]
c_nodiv = bsm(50.0, 50.0, 0.04, 0.30, 0.5)[2]
show("call on S*", c_div, 3.83, 2)
show("call on a stock paying nothing", c_nodiv, 4.70, 2)
show("value the dividends cost the call, percent", 100 * (1 - c_div / c_nodiv), 18.5, 1)

print()
ratio6 = 1 - exp(-0.06 * 2)
show("Appendix A, 6% yield, two years: PV(D)/S", ratio6, 0.113, 3)
show("Appendix A: sigma*/sigma - 1, percent", 100 * (1 / (1 - ratio6) - 1), 13, 0)
ratio4 = 1 - exp(-0.04 * 5)
print("Question 15.16, 4%% yield, five years: PV(D)/S = %.4f, sigma*/sigma - 1 = %.1f%%"
      % (ratio4, 100 * (1 / (1 - ratio4) - 1)))
sig_star = 0.30 / (1 - ratio4)
S5 = 50.0 * (1 - ratio4)
print("five-year at-the-money call on S* = %.4f: %.4f at sigma = 30%%, %.4f at sigma* = %.2f%%"
      % (S5, bsm(S5, 50.0, 0.04, 0.30, 5.0)[2], bsm(S5, 50.0, 0.04, sig_star, 5.0)[2], 100 * sig_star))

## 9. The American put: the European floor against a tree — Section 15.10

The formula gives a floor, $P \ge p$, and a lattice gives the American value. For
the two-year put of Section 13.9's convergence figure ($S_0 = 50$, $K = 52$,
$r = 5\%$, $\sigma = 30\%$), the floor is 6.7601 and a 2,000-step tree gives
7.4719, an early-exercise premium of 10.5%. For one-month out-of-the-money puts
on the same stock the premium is under a percent. The figure shows where the
floor is slack: deep in the money, where the American put is worth exercising
and sits on intrinsic value while the European put falls below it.

In [ ]:
def crr_put(S, K_, r_, sig, t, steps, american=True):
    """Put on a Cox-Ross-Rubinstein tree; NumPy rolls each time slice back at once."""
    dt = t / steps
    up = exp(sig * sqrt(dt)); dn = 1.0 / up
    pu = (exp(r_ * dt) - dn) / (up - dn); disc = exp(-r_ * dt)
    j = np.arange(steps + 1)
    v = np.maximum(K_ - S * up ** j * dn ** (steps - j), 0.0)
    for i in range(steps - 1, -1, -1):
        v = disc * (pu * v[1:] + (1 - pu) * v[:-1])
        if american:
            j = np.arange(i + 1)
            v = np.maximum(v, K_ - S * up ** j * dn ** (i - j))
    return v[0]


euro_put = bsm(50.0, 52.0, 0.05, 0.30, 2.0)[3]
amer_put = crr_put(50.0, 52.0, 0.05, 0.30, 2.0, 2000)
show("European put, the formula", euro_put, 6.7601)
show("American put, 2,000-step tree", amer_put, 7.4719)
show("early-exercise premium, percent", 100 * (amer_put / euro_put - 1), 10.5, 1)
for strike in (45.0, 48.0):
    e = bsm(50.0, strike, 0.05, 0.30, 1 / 12)[3]
    a = crr_put(50.0, strike, 0.05, 0.30, 1 / 12, 2000)
    print("one-month put struck at %.0f: European %.4f, American %.4f, premium %.3f%%"
          % (strike, e, a, 100 * (a / e - 1)))

spots = np.linspace(25.0, 80.0, 56)
fig, ax = plt.subplots()
ax.plot(spots, np.maximum(52.0 - spots, 0.0), color="0.6", ls="--", lw=1.2, label="intrinsic value")
ax.plot(spots, [crr_put(s, 52.0, 0.05, 0.30, 2.0, 400) for s in spots], color=BLUE, lw=1.6,
        label="American (400-step tree)")
ax.plot(spots, [bsm(s, 52.0, 0.05, 0.30, 2.0)[3] for s in spots], color=AMBER, ls="-.", lw=1.6,
        label="European (formula)")
ax.set_xlabel("Stock price S0"); ax.set_ylabel("Put value")
ax.set_title("The European floor under a two-year American put")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 10. Black's approximation — Example 15.6

The greater of two European calls: one to $T$ on the dividend-adjusted price,
one to the ex-dividend date on the unadjusted price. The tree that checks it
holds the dividend in escrow: it models $S^* = S - \text{PV}(D)$ on a
Cox–Ross–Rubinstein lattice and adds back the dividend's present value at every
node up to and including the ex-dividend date, where the holder may exercise
cum-dividend. With exercise switched off the tree must reproduce the European
value on $S^*$, which confirms it before it is trusted.

The approximation chooses between the two policies today; the holder chooses at
the dividend date after seeing the price. That option to decide later is what it
omits, so the tree is always worth more, here by a few cents, as the chapter
says.

In [ ]:
S_b, K_b, r_b, sig_b, T_b, D, t1 = 52.0, 50.0, 0.04, 0.30, 0.5, 1.50, 5 / 12

show("K(1 - e^{-r(T - t1)})", K_b * (1 - exp(-r_b * (T_b - t1))), 0.17, 2)
pvD = D * exp(-r_b * t1)
show("PV(D)", pvD, 1.4752)
show("S* = S0 - PV(D)", S_b - pvD, 50.5248)
hold = bsm(S_b - pvD, K_b, r_b, sig_b, T_b)[2]
early = bsm(S_b, K_b, r_b, sig_b, t1)[2]
show("European to T on S*", hold, 5.00, 2)
show("European to t1 on S0", early, 5.48, 2)
black = max(hold, early)
show("Black's approximation", black, 5.48, 2)


def escrowed_call(S, K_, r_, sig, t, div, t_div, steps, american=True):
    """Call on a stock paying div at t_div: CRR tree on S* plus the dividend's PV up to t_div."""
    dt = t / steps
    up = exp(sig * sqrt(dt)); dn = 1.0 / up
    pu = (exp(r_ * dt) - dn) / (up - dn); disc = exp(-r_ * dt)
    i_div = int(round(t_div / dt))
    base = S - div * exp(-r_ * t_div)

    def stock(i):
        j = np.arange(i + 1)
        level = base * up ** j * dn ** (i - j)
        return level + div * exp(-r_ * (t_div - i * dt)) if i <= i_div else level

    v = np.maximum(stock(steps) - K_, 0.0)
    for i in range(steps - 1, -1, -1):
        v = disc * (pu * v[1:] + (1 - pu) * v[:-1])
        if american:
            v = np.maximum(v, stock(i) - K_)
    return v[0]


print()
print("%8s %14s %14s %14s" % ("steps", "European tree", "American tree", "tree - Black"))
for steps in (600, 1200, 3000):
    eu = escrowed_call(S_b, K_b, r_b, sig_b, T_b, D, t1, steps, american=False)
    am = escrowed_call(S_b, K_b, r_b, sig_b, T_b, D, t1, steps)
    print("%8d %14.4f %14.4f %11.2f cents" % (steps, eu, am, 100 * (am - black)))
print("the European tree converges to %.4f, the formula on S*; Black understates the American tree"
      % hold)

## 11. The skew of Table 15.3 — Section 15.11

The call mids of Table 10.2, inverted with the two clocks set apart above: variance
over 31 trading days, the strike and the \$1.00 dividend of Section 11.6
discounted over 45 calendar days, $S_0 = 48.30$, $r = 5\%$. If the model were
right, all six implied volatilities would be equal. They fall from 56.9% to
33.2%, a factor of 1.7. The calls are American, so part of each figure, most at
the lowest strikes, is early-exercise premium folded in, as the chapter says.

The last table prices the whole chain at the single volatility read from the
47.50 strike, nearest the money, which is what the chapter means by using a
model the market has already rejected: the low strikes come out cheap and the
high strikes rich. That comparison is the notebook's, not the chapter's.

In [ ]:
S_c, r_c = 48.30, 0.05
T_trd, T_cal = 31 / 252, 45 / 365
S_adj = S_c - 1.00 * exp(-r_c * T_cal)
TABLE_15_3 = ((42.50, -12.0, 6.650, 56.9), (45.00, -6.8, 4.425, 46.0), (47.50, -1.7, 2.670, 39.7),
              (50.00, 3.5, 1.475, 36.7), (52.50, 8.7, 0.710, 34.5), (55.00, 13.9, 0.305, 33.2))

print("%8s %10s %10s %12s %12s" % ("strike", "moneyness", "call mid", "implied vol", "book"))
ivs, money = [], []
for strike, mny, mid, book in TABLE_15_3:
    iv = brentq(lambda v: call_two_clocks(S_adj, strike, r_c, v, T_trd, T_cal) - mid, 0.01, 3.0)
    ivs.append(iv); money.append(100 * (strike / S_c - 1))
    print("%8.2f %9.1f%% %10.3f %11.1f%% %11.1f%%" % (strike, money[-1], mid, 100 * iv, book))
    show("  K = %.2f moneyness, percent" % strike, money[-1], mny, 1)
    show("  K = %.2f implied volatility, percent" % strike, 100 * ivs[-1], book, 1)
show("ratio of the extremes", ivs[0] / ivs[-1], 1.7, 1)

flat = ivs[2]
print()
print("%8s %10s %14s %12s" % ("strike", "call mid", "at one sigma", "model - mid"))
for (strike, _, mid, _), iv in zip(TABLE_15_3, ivs):
    model = call_two_clocks(S_adj, strike, r_c, flat, T_trd, T_cal)
    print("%8.2f %10.3f %14.3f %+12.3f" % (strike, mid, model, model - mid))

fig, ax = plt.subplots(figsize=(5.4, 3.2))
ax.plot(money, [100 * v for v in ivs], color=BLUE, lw=1.6, marker="o", ms=4, label="market")
ax.axhline(100 * flat, color=AMBER, ls="--", lw=1.4, label="model: one sigma")
ax.set_xlabel("Moneyness K/S0 - 1 (%)"); ax.set_ylabel("Implied volatility (%)")
ax.set_title("Table 15.3: implied volatility across strikes")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()